# Chapter 1 — Introduction and Overview
## Foundations of Sensitivity Analysis — Arriola & Hyman (SIAM, 2026)

**Purpose:** Introduce the SA question, the POI/QOI vocabulary, and preview the
three unifying ideas through interactive examples.

**Key claims tested:**
- Forward SA (`What if?`) and adjoint SA (`How come?`) answer complementary questions
- Local SA can mislead when uncertainty is large (Example 1.1: q = p²)
- The sensitivity index is dimensionless — raw derivatives are not comparable across parameters

**Authors:** Leon M. Arriola & James M. Hyman (mhyman@tulane.edu)  
**Repository:** https://github.com/machyman/arriola2026sa

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
np.random.seed(42)

# ── Scale switch ───────────────────────────────────────────────────────────────
# FULL = False → quick verification (< 2 min on CPU)
# FULL = True  → publication-quality figures
FULL = False
N_PLOT = 500 if FULL else 200

print('Chapter 1: Introduction and Overview')
print(f'Scale: {"FULL" if FULL else "QUICK"}, N_PLOT={N_PLOT}')

In [ ]:
# ── Verification Suite ─────────────────────────────────────────────────────────
print('=' * 60)
print('VERIFICATION SUITE')
print('=' * 60)

# Test 1: Example 1.1 — local SI for q = p² is constant = 2
def q(p): return p**2
def SI_analytic(p): return (p / q(p)) * 2*p  # = 2 always

for p_test in [0.1, 0.5, 1.0, 2.0]:
    si = SI_analytic(p_test)
    assert abs(si - 2.0) < 1e-12, f'FAIL: SI={si:.6f} at p={p_test}'
print('Test 1 PASS: SI for q=p² is constant = 2 at all p')

# Test 2: global range is 100× despite constant local SI
q_low, q_high = q(0.01), q(1.0)
ratio = q_high / q_low
assert abs(ratio - 10000.0) < 1e-6, f'FAIL: ratio={ratio}'
print(f'Test 2 PASS: q range = {q_low:.4f} to {q_high:.4f} (ratio = {ratio:.0f}×)')

# Test 3: SIR R₀ = kβτ, all SIs = 1
c, beta, tau_R = 5.0, 0.06, 7.0
R0 = c * beta * tau_R
SI_c    = (c/R0) * (beta*tau_R)
SI_beta = (beta/R0) * (c*tau_R)
SI_tau_R  = (tau_R/R0) * (c*beta)
for name, val in [('c',SI_c),('β',SI_beta),('τ',SI_tau_R)]:
    assert abs(val - 1.0) < 1e-12, f'FAIL: SI_{name}={val}'
print(f'Test 3 PASS: R₀={R0:.2f}, SI_c=SI_β=SI_τ=1 (all exact)')

print(f'\nAll 3 verification tests PASSED.')
print('=' * 60)

In [ ]:
# ── Figure 1: Local SI is constant but global behavior varies ─────────────────
# Reproduces Example 1.1 from §1.3
p_range = np.linspace(0.01, 1.5, N_PLOT)
q_vals  = p_range**2
p_star  = 0.1

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: q(p) = p² with tangent line at p* = 0.1
ax = axes[0]
ax.plot(p_range, q_vals, 'steelblue', lw=2, label=r'$q(p) = p^2$')
dq = 2*p_star  # derivative at p*
tangent = q(p_star) + dq*(p_range - p_star)
ax.plot(p_range, np.clip(tangent, -0.1, 2.5), 'coral', lw=1.5,
        ls='--', label=fr'Tangent at $p^*={p_star}$')
ax.axvline(p_star, color='gray', ls=':', lw=1)
ax.axvline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel(r'$p$', fontsize=12)
ax.set_ylabel(r'$q(p)$', fontsize=12)
ax.set_title('The function and its local linearization', fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(0, 1.5); ax.set_ylim(-0.05, 2.0)
ax.annotate(fr'$p^*={p_star}$, $S_p^q=2$', xy=(p_star, q(p_star)),
            xytext=(0.35, 0.3), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='gray'))

# Right: SI as a function of p* — showing it is constant = 2
ax2 = axes[1]
si_vals = np.full_like(p_range, 2.0)
ax2.plot(p_range, si_vals, 'steelblue', lw=2, label=r'$\mathcal{S}_p^q = 2$ (exact)')
q_range_ratio = q_vals[-1] / q_vals[0]
ax2.set_xlabel(r'Nominal point $p^*$', fontsize=12)
ax2.set_ylabel(r'$\mathcal{S}_{p}^{q}$', fontsize=12)
ax2.set_title(
    f'Local SI is constant = 2\n'
    fr'yet $q$ ranges {q_range_ratio:.0f}× over $p\in[0.01,1.5]$', fontsize=11)
ax2.axhline(2, color='coral', ls='--', lw=1.5, label='SI = 2')
ax2.set_ylim(0, 3); ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('ch01_fig1_local_vs_global.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch01_fig1_local_vs_global.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Figure 2: SIR R₀ sensitivity index — forward vs adjoint cost ──────────────
# Illustrates Fig 1.2 from §1.2
m_vals = np.arange(1, 21)   # number of POIs
r_vals = [1, 3, 5]          # number of QOIs

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ['steelblue', 'coral', 'seagreen']
for r, col in zip(r_vals, colors):
    ax.axhline(r, color=col, ls='--', lw=1.5, alpha=0.7,
               label=f'Adjoint: {r} solve{"s" if r>1 else ""} (r={r} QOIs)')

ax.plot(m_vals, m_vals, 'c-', lw=2, label='Forward SA: m solves (1 per POI)')

ax.set_xlabel('Number of POIs ($m$)', fontsize=12)
ax.set_ylabel('Number of model solves', fontsize=12)
ax.set_title('Forward vs Adjoint SA — Computational Cost\n'
             '(adjoint cost is independent of $m$)', fontsize=11)
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(1, 20); ax.set_ylim(0, 21)
ax.grid(True, alpha=0.3)

# Annotate crossover points
for r, col in zip(r_vals, colors):
    ax.annotate(f'Equal cost\nat m=r={r}', xy=(r, r),
                xytext=(r+1, r+2.5), fontsize=8,
                arrowprops=dict(arrowstyle='->', color=col, lw=0.8),
                color=col)

plt.tight_layout()
plt.savefig('ch01_fig2_cost_comparison.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch01_fig2_cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── Results summary ────────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 1,
    'title': 'Introduction and Overview',
    'verification_tests': 3,
    'SIR_R0': {'c': 5.0, 'beta': 0.06, 'tau_R': 7.0, 'R0': R0,
               'SI_c': SI_c, 'SI_beta': SI_beta, 'SI_tau_R': SI_tau_R},
    'example_1_1': {'SI_constant': 2.0, 'global_range_ratio': ratio},
    'figures': ['ch01_fig1_local_vs_global.pdf', 'ch01_fig2_cost_comparison.pdf']
}
for k_r, v in RESULTS.items():
    print(f'  {k_r}: {v}')

In [ ]:
# ── Download outputs ───────────────────────────────────────────────────────────
try:
    from google.colab import files
    for f in RESULTS['figures']:
        files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')
    for f in RESULTS['figures']:
        import os
        print(f'  {f}: {os.path.getsize(f)//1024} KB' if os.path.exists(f) else f'  {f}: not found')